# Домашнє завдання: Інтерактивні візуалізації з Plotly

## Опис завдання
У цьому домашньому завданні ви будете створювати інтерактивні візуалізації з допомогою бібліотеки Plotly. Ви дізнаєтесь різницю між Plotly Express (швидкі графіки) та Graph Objects (повний контроль), та створите інтерактивний дашборд.

**Опис колонок:**
- `datetime` - дата та час
- `season` - квартал (1-Q1, 2-Q2, 3-Q3, 4-Q4)
- `holiday` - чи є день святковим (0=ні, 1=так)
- `workingday` - чи є день робочим (0=ні, 1=так)
- `weather` - погодні умови (1=ясно, 2=туман, 3=легкий дощ, 4=сильний дощ)
- `temp` - температура в градусах Цельсія
- `atemp` - відчувається як температура
- `humidity` - вологість (%)
- `windspeed` - швидкість вітру
- `casual` - кількість випадкових користувачів
- `registered` - кількість зареєстрованих користувачів
- `count` - загальна кількість орендованих велосипедів

## Підготовка даних

---

🌱 Коментар щодо сезонності

Колонка season у датасеті представляє саме квартали року, а не метеорологічні сезони. Тому всі аналізи сезонності ви можете будувати на основі кварталів.

Водночас дані були зібрані в Індії, де поділ на сезони інший, ніж у Європі чи США. Якщо ви хочете дослідити сезонність відповідно до індійської системи сезонів, можна створити окрему колонку.

Справжні сезони в Індії:

| Сезон        | Місяці                     |
| ------------ | -------------------------- |
| Winter       | December–February (12,1,2) |
| Summer       | March–May (3,4,5)          |
| Monsoon      | June–September (6,7,8,9)   |
| Post-monsoon | October–November (10,11)   |


Тоді потрібно зробити нову колонку weather_season_india, мапнувши місяці так:

12, 1, 2 → 1 (Winter)

3, 4, 5 → 2 (Summer)

6–9 → 3 (Monsoon)

10–11 → 4 (Post-Monsoon)


In [1]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

DATA_URL = 'https://drive.google.com/uc?export=download&id=1abUBbpxK6GOoUgA5f40L7EA9ROEiTuf0'
for path in ['../data/yulu_rental.csv', 'yulu_rental.csv', DATA_URL]:
    if path.startswith('http') or os.path.exists(path):
        break
df = pd.read_csv(path)
df['datetime'] = pd.to_datetime(df['datetime'])

df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.day_name()

quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
df['quarter_name'] = df['season'].map(quarter_map)
print(df.shape)

(10886, 16)


## Завдання 1: Базовий інтерактивний лінійний графік (Plotly Express)

**Завдання:**
Створіть інтерактивний лінійний графік динаміки оренди за часом (рівень деталізації - як в даних) з можливістю zoom та hover.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яка перевага інтерактивного графіка над статичним?
2. Чому на графіку є "пробіли" - ділянки, де одна пряма лінія зʼєднує два "суцільних" блоки з даними? Як би ви це могли дослідити на статичному графіку?


In [2]:
fig1 = px.line(
    df, x='datetime', y='count',
    title='Погодинна динаміка оренди велосипедів, 2011–2012',
    labels={'datetime': 'Дата і час', 'count': 'Кількість оренд за годину'},
)
fig1.update_traces(line=dict(width=1))
fig1.update_xaxes(rangeslider_visible=True)
fig1.show()

print('Дні місяця в даних:', sorted(int(d) for d in df['datetime'].dt.day.unique()))

Дні місяця в даних: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


**Відповіді:**

1. Інтерактивний графік дає змогу наблизити будь-який період (зумом або повзунком), навести курсор і побачити точне значення та час, приховати зайве. На статичному графіку 10 886 погодинних точок зливаються в суцільну пляму: щоб роздивитися окремий тиждень чи день, довелося б будувати окремі графіки.
2. У даних є лише 1–19 числа кожного місяця (див. вивід вище): з 20-го числа до кінця місяця записів немає. Plotly зʼєднує останню годину 19-го числа з першою годиною 1-го числа наступного місяця прямою лінією — це і є «пробіли». На статичному графіку це можна дослідити так: порахувати кількість записів по датах (`df['datetime'].dt.date.value_counts()`), побудувати графік кількості спостережень за день або перетворити ряд на регулярну погодинну сітку (`resample('h')`). Тоді пропуски стануть порожніми проміжками без лінії.

## Завдання 2: Scatter plot з додатковими даними (Plotly Express)

**Завдання:**
Створіть scatter plot кількості орендованих велосипедів випадковими користувачами vs кількості орендованих велосипедів зареєстрованими користувачами. Розмір точок встановіть за сумарною кількістю велосипедів, які були взяті в оренду, а колір - за сезоном(кварталом). В hover_data - додайте деталі, які допоможуть вам в подальшому аналізі.

Дослідіть графік. Зверніть увагу, що ви можете вмикати і вимикати окремі квартали, якщо будете клікати на колір кварталу в легенді графіку.

**Дайте відповідь на питання.**
- Як ви проінтерпретуєте роздвоєність цього графіку (дві явні лінії)? Що це означає?
- Які висновки для компанії, які дає велосипеди в оренду, ви можете зробити з цього графіку? 3 основних висновки.

In [3]:
fig2 = px.scatter(
    df, x='casual', y='registered',
    size='count', color='quarter_name',
    category_orders={'quarter_name': ['Q1', 'Q2', 'Q3', 'Q4']},
    hover_data={'datetime': True, 'hour': True, 'weekday': True, 'workingday': True,
                'holiday': True, 'weather': True, 'temp': True, 'count': True},
    opacity=0.6, size_max=15,
    title='Випадкові vs зареєстровані користувачі (розмір — усього оренд, колір — квартал)',
    labels={'casual': 'Оренди випадкових користувачів', 'registered': 'Оренди зареєстрованих користувачів',
            'quarter_name': 'Квартал'},
)
fig2.show()

print(df.groupby('workingday')[['casual', 'registered']].mean().round(1))
print('Частка випадкових користувачів у всіх орендах:', round(df['casual'].sum() / df['count'].sum(), 3))

            casual  registered
workingday                    
0             59.3       129.2
1             25.1       167.9
Частка випадкових користувачів у всіх орендах: 0.188


**Відповіді:**

- **Роздвоєність.** Це два різні режими користування. Крута гілка — робочі дні: велосипеди беруть переважно зареєстровані користувачі, щоб доїхати на роботу чи навчання. Полога гілка — вихідні й свята: випадкових користувачів значно більше. Таблиця вище це підтверджує: у робочі дні на годину припадає в середньому 25 випадкових і 168 зареєстрованих оренд (співвідношення 0,15), у неробочі — 59 і 129 (0,46). Якщо розфарбувати графік за `workingday`, гілки розділяються.
- **Три висновки для компанії:**
  1. Зареєстровані користувачі дають близько 81 % усіх оренд. Це основа доходу: важливо утримувати їх і мати достатньо велосипедів у пікові години робочих днів (8:00 та 17:00–18:00).
  2. У вихідні та теплі квартали різко зростає частка випадкових користувачів. Це шанс конвертувати їх у зареєстрованих: знижка на підписку після кількох поїздок, туристичні тарифи та тарифи вихідного дня.
  3. Сезонність сильна: у Q1 середня кількість оренд на годину 116, у Q3 — 234. Узимку варто планувати обслуговування парку, а влітку і восени — збільшувати кількість доступних велосипедів.

## Завдання 3: Порівняння Plotly Express vs Graph Objects

**Завдання:**
Створіть лінійний графік помісячної динаміки оренди (кількість оренд) велосипедів двома способами - з Plotly Express та з Graph Objects.

**Дайте відповіді на питання.**
1. Як ви розумієте основну різницю між цими двома підходами?
2. Коли краще використовувати Plotly Express?
3. Коли потрібен Graph Objects?


In [4]:
monthly = df.set_index('datetime')['count'].resample('MS').sum().reset_index()
monthly.columns = ['month_start', 'total_count']

fig3_px = px.line(monthly, x='month_start', y='total_count', markers=True,
                  title='Помісячна кількість оренд — Plotly Express',
                  labels={'month_start': 'Місяць', 'total_count': 'Кількість оренд'})
fig3_px.show()

fig3_go = go.Figure()
fig3_go.add_trace(go.Scatter(
    x=monthly['month_start'], y=monthly['total_count'],
    mode='lines+markers', name='Кількість оренд',
    line=dict(color='#2a9d8f', width=3), marker=dict(size=7),
    hovertemplate='%{x|%b %Y}: %{y:,} оренд<extra></extra>',
))
fig3_go.update_layout(title='Помісячна кількість оренд — Graph Objects',
                      xaxis_title='Місяць', yaxis_title='Кількість оренд',
                      template='plotly_white', hovermode='x unified')
fig3_go.show()

by_year = monthly.groupby(monthly['month_start'].dt.year)['total_count'].sum()
print('Оренд за рік:', by_year.to_dict(), '| зростання 2012 до 2011:', round(by_year[2012] / by_year[2011] - 1, 3))

Оренд за рік: {2011: 781979, 2012: 1303497} | зростання 2012 до 2011: 0.667


**Відповіді:**

1. **Різниця.** Plotly Express — високорівнева обгортка: одна функція бере DataFrame і назви колонок і одразу будує готову фігуру з розумними налаштуваннями. Graph Objects — низькорівневий інтерфейс: фігуру збираємо вручну з окремих трейсів (`go.Scatter`, `go.Bar`…) і налаштувань `layout`. Express під капотом створює ті самі обʼєкти Graph Objects, тому результат Express можна донастроїти методами `update_*`.
2. **Plotly Express** — для швидкого дослідження даних і типових графіків із «охайного» DataFrame: коли треба за хвилину побачити розподіл, тренд чи розбивку за категоріями (`color`, `facet_col`, `animation_frame`).
3. **Graph Objects** — коли потрібен повний контроль: кілька різнотипних трейсів на одному графіку, дві осі Y, дашборди через `make_subplots`, власні `hovertemplate`, анотації, кнопки та слайдери.

Про саму динаміку: у 2012 році оренд на 67 % більше, ніж у 2011; піки — влітку й на початку осені, мінімум — у січні.

## Завдання 4 (Опціональне): Дашборд з make_subplots (Graph Objects)

**Завдання:**
Створіть дашборд з 4 різними графіками в одній фігурі:
- Bar chart - середні значення загальної кількості оренд велосипедів за сезонами(кварталами)
- Pie chart - відсоткове співвідношення погодних умов в даних
- Line chart - середнє значення загальної кількості оренд велосипедів за годинами протягом доби
- Scatter plot - кореляція температури vs вологість

Додайте заголовок на дашборд.

**Дайте відповідь на питання**
- На ваш погляд, яка перевага об'єднання графіків в один дашборд?

In [5]:
season_mean = df.groupby('quarter_name')['count'].mean().reindex(['Q1', 'Q2', 'Q3', 'Q4']).round(1)
weather_names = {1: 'Ясно', 2: 'Туман / хмарно', 3: 'Легкий дощ / сніг', 4: 'Сильний дощ'}
weather_share = df['weather'].map(weather_names).value_counts()
hour_mean = df.groupby('hour')['count'].mean().round(1)

fig4 = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'xy'}, {'type': 'domain'}], [{'type': 'xy'}, {'type': 'xy'}]],
    subplot_titles=('Середня кількість оренд за кварталами', 'Погодні умови, частка годин',
                    'Середня кількість оренд за годинами доби', 'Температура vs вологість'),
)
fig4.add_trace(go.Bar(x=season_mean.index, y=season_mean.values, marker_color='#2a9d8f',
                      name='Середнє за квартал', text=season_mean.values, textposition='outside'),
               row=1, col=1)
fig4.add_trace(go.Pie(labels=weather_share.index, values=weather_share.values, hole=0.35,
                      name='Погода', textinfo='percent+label'), row=1, col=2)
fig4.add_trace(go.Scatter(x=hour_mean.index, y=hour_mean.values, mode='lines+markers',
                          line=dict(color='#e76f51'), name='Середнє за годину'), row=2, col=1)
fig4.add_trace(go.Scatter(x=df['temp'], y=df['humidity'], mode='markers',
                          marker=dict(size=4, opacity=0.3, color='#264653'), name='Година спостереження'),
               row=2, col=2)
fig4.update_xaxes(title_text='Квартал', row=1, col=1)
fig4.update_yaxes(title_text='Оренд за годину', row=1, col=1)
fig4.update_xaxes(title_text='Година доби', row=2, col=1)
fig4.update_yaxes(title_text='Оренд за годину', row=2, col=1)
fig4.update_xaxes(title_text='Температура, °C', row=2, col=2)
fig4.update_yaxes(title_text='Вологість, %', row=2, col=2)
fig4.update_layout(title_text='Дашборд: оренда велосипедів Yulu, 2011–2012', height=800,
                   showlegend=False, template='plotly_white')
fig4.show()

print('Кореляція температури та вологості:', round(df['temp'].corr(df['humidity']), 3))

Кореляція температури та вологості: -0.065


**Відповідь:** дашборд показує кілька зрізів одночасно на одному екрані. Їх легше порівнювати і повʼязувати між собою: наприклад, найбільше оренд у теплих Q2–Q3, переважно ясна погода (66 % годин), а протягом доби видно ранковий і вечірній піки. Такий дашборд зручно показати керівнику чи клієнту і зберегти одним HTML-файлом. Окремо видно, що температура і вологість майже не повʼязані (кореляція близько −0,07).

## Завдання 5 (Опціональне): 3D візуалізація

**Завдання:**
Створіть 3D scatter plot для аналізу взаємозв'язку температури, швидкості вітру та загальної кількості орендованих велосипедів. Колір встановіть за сезоном(кварталом), а розмір - за загальною кількість оренд також.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яку додаткову інформацію, на ваш погляд, дає 3D візуалізація?
2. Чи видно кластери в 3D просторі?
3. Чи ви можете зробити висновки з цієї візуалізації, чи вам було простіше побудувати кілька 2D?



In [6]:
fig5 = px.scatter_3d(
    df, x='temp', y='windspeed', z='count',
    color='quarter_name', size='count', size_max=18, opacity=0.6,
    category_orders={'quarter_name': ['Q1', 'Q2', 'Q3', 'Q4']},
    labels={'temp': 'Температура, °C', 'windspeed': 'Швидкість вітру', 'count': 'Кількість оренд',
            'quarter_name': 'Квартал'},
    title='Температура, вітер і кількість оренд (3D)',
)
fig5.update_layout(height=700)
fig5.show()

print('Унікальних значень швидкості вітру:', df['windspeed'].nunique(),
      '| частка нулів:', round((df['windspeed'] == 0).mean(), 3))
print('Середня температура за кварталами:', df.groupby('quarter_name')['temp'].mean().round(1).to_dict())
print('Кореляція з кількістю оренд: температура', round(df['temp'].corr(df['count']), 2),
      '| вітер', round(df['windspeed'].corr(df['count']), 2))

Унікальних значень швидкості вітру: 28 | частка нулів: 0.121
Середня температура за кварталами: {'Q1': 12.5, 'Q2': 22.8, 'Q3': 28.8, 'Q4': 16.6}
Кореляція з кількістю оренд: температура 0.39 | вітер 0.1


**Відповіді:**

1. 3D-графік показує одночасно три числові змінні (температуру, вітер і кількість оренд), а колір додає четверту — квартал. Видно, що кількість оренд росте з температурою, а від вітру майже не залежить (кореляція 0,39 проти 0,10).
2. Кластери є, але за температурою, а не за вітром: точки Q1 зібрані в холодній частині (середня температура 12,5 °C) з меншою кількістю оренд, Q3 — у теплій (28,8 °C) з найвищими значеннями. Уздовж осі вітру точки лягають «шарами», бо в даних лише 28 різних значень швидкості вітру, а 12 % значень дорівнюють нулю (схоже на пропуски).
3. Висновки з 3D робити складніше: графік треба обертати, точки перекривають одна одну, важко зчитати значення. Ті самі висновки простіше отримати з кількох 2D-графіків: кількість vs температура, кількість vs вітер, розбивка за кварталами. 3D корисний для дослідження, а для презентації краще 2D.


## Завдання 6: Експорт та збереження інтерактивних графіків

**Завдання:**
Збережіть побудований попередній графік на plotly в формат HTML. Також змініть вручну щось на графіку (зум, виділення частини графіку) і збережіть його як статичне зображення через іконку фотоапарату у формат PNG. Завантажте файли з графіком у HTML та PNG (або посилання на них на github) разом з посиланням на цей ноутбук при здачі ДЗ.


In [7]:
fig5.write_html('plotly_3d_chart.html', include_plotlyjs='cdn')
print('Збережено: plotly_3d_chart.html')

Збережено: plotly_3d_chart.html


**Файли для здачі:**

- `plotly_3d_chart.html` — інтерактивний 3D-графік із завдання 5.
- `plotly_3d_chart_zoom.png` — той самий графік після ручної зміни: наближено й повернуто камеру, потім збережено кнопкою з фотоапаратом (Download plot as a png).

Обидва файли лежать поруч із цим ноутбуком.